In [1]:
# @title Setup (chạy ô này trước)
# Colab bắt đầu với một máy trống — clone repo và cài dependency.
import os, subprocess, sys

REPO = "https://github.com/VinUni-AI20k/Day21-Track3-Finetuning-Lab.git"
if not os.path.exists("Day21-Track3-Finetuning-Lab"):
    subprocess.run(["git", "clone", "-q", REPO], check=True)
os.chdir("Day21-Track3-Finetuning-Lab")
sys.path.insert(0, "src")

# Install from requirements.txt, NOT a copied list. The copied list is how the
# torchao>=0.16 pin reached requirements.txt and this bootstrap on different days --
# and a bootstrap missing a pin does not fail here, it fails 10 minutes later inside
# get_peft_model(). One source of truth. torch is preinstalled on Colab and
# requirements.txt pins it compatibly, so that line is a no-op.
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"],
               check=True)

os.environ.setdefault("COMPUTE_TIER", "T4")
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE — Runtime > Change runtime type > T4 GPU")


GPU: Tesla T4


## Bonus B1 (+3) và B5 (+2)
Run All trên Colab web T4. Khôi phục correct và attn_only từ ZIP Drive, đo trước/sau merge trên 50 ticket và hoán đổi hai adapter 16-bit trên cùng base. Phần cuối upload adapter correct công khai lên HF Hub, token chỉ nhập qua getpass. Chưa khai báo điểm thưởng trong report đến khi có artifact và link xác minh.


In [2]:
# @title Khôi phục bonus từ Drive, giữ cùng base model
from pathlib import Path
ROOT = Path("/content/Day21-Track3-Finetuning-Lab")
os.chdir(ROOT)
os.environ["COMPUTE_TIER"]="T4"
os.environ["EVAL_LIMIT"]="0"
os.environ.pop("BASE_MODEL",None)
assert torch.cuda.is_available(), "Bonus NB6 cần T4 GPU."
assert torch.cuda.mem_get_info()[0] >= 10 * 1024**3, "Dừng kernel GPU khác và restart session để giải phóng model cũ."
# Tự khôi phục backup công khai — tài khoản khác không cần mount Drive.
from pathlib import PurePosixPath
from zipfile import ZipFile,is_zipfile
import shutil
BACKUP_FILE_ID = "1FY0vRKlqnreS-JV0D4-DxanrUGMI9R8p"
BACKUP_FOLDER_URL = "https://drive.google.com/drive/folders/18wN-UeRbCC6YQk1lYVV7v0HTdt8wrSOk"
BACKUP_SIZE = 481443440
required = ["results/verdict.json","results/baselines_frozen.json","data/split/train.jsonl",
            "adapters/correct/adapter_model.safetensors","adapters/correct/adapter_config.json", "adapters/attn_only/adapter_model.safetensors", "adapters/attn_only/adapter_config.json"]
if not all((ROOT / name).is_file() for name in required):
    subprocess.run([sys.executable,"-m","pip","install","-q","gdown>=5,<6"],check=True)
    import gdown
    archive_path=Path("/content/lab21_shared_backup.zip")
    if not archive_path.is_file() or archive_path.stat().st_size != BACKUP_SIZE:
        downloaded=gdown.download(id=BACKUP_FILE_ID,output=str(archive_path),quiet=False)
        assert downloaded, "Không tải được ZIP: kiểm tra quyền Anyone with the link hoặc giới hạn tải của Drive."
    assert archive_path.stat().st_size == BACKUP_SIZE, "ZIP chưa tải đủ hoặc file Drive đã thay đổi."
    assert is_zipfile(archive_path), "Nội dung tải về không phải ZIP."
    with ZipFile(archive_path) as archive:
        assert archive.testzip() is None, "ZIP bị lỗi CRC."
        for entry in archive.infolist():
            parts=PurePosixPath(entry.filename).parts
            assert ".." not in parts and not PurePosixPath(entry.filename).is_absolute(), "ZIP có đường dẫn không an toàn."
            positions=[i for i,x in enumerate(parts) if x in ("results","adapters","data")]
            if not positions: continue
            relative=PurePosixPath(*parts[positions[0]:])
            allowed=relative.parts[0] == "results" or relative.parts[:2] in (("data","split"),("adapters","correct"),("adapters","attn_only"))
            if not allowed or entry.is_dir(): continue
            assert (entry.external_attr >> 16) & 0o170000 != 0o120000, "ZIP có symlink."
            target=(ROOT / str(relative)).resolve()
            assert ROOT.resolve() in target.parents
            target.parent.mkdir(parents=True,exist_ok=True)
            with archive.open(entry) as source,target.open("wb") as dest:
                shutil.copyfileobj(source,dest)
assert all((ROOT / name).is_file() for name in required), "Backup thiếu artifact cần thiết."
print("Đã có baseline và adapter correct. Bước tiếp theo chỉ inference, không train lại.")


Downloading...
From (original): https://drive.google.com/uc?id=1FY0vRKlqnreS-JV0D4-DxanrUGMI9R8p
From (redirected): https://drive.google.com/uc?id=1FY0vRKlqnreS-JV0D4-DxanrUGMI9R8p&confirm=t&uuid=1fd0e1e7-1af3-4104-8a29-0b3a2920d73d
To: /content/lab21_shared_backup.zip
100%|██████████| 481M/481M [00:02<00:00, 227MB/s]


Đã có baseline và adapter correct. Bước tiếp theo chỉ inference, không train lại.


# NB6 (tuỳ chọn) — Merge, kiểm chứng sau merge, và hoán đổi adapter

Deck §23. Hai đường triển khai:
* **Merge** — `W = W₀ + (α/r)·BA`, đồ thị phục vụ giống hệt base → **không** overhead.
* **Giữ riêng** — một base trong VRAM, nhiều adapter, chọn theo từng request.

> Ngày 20 sở hữu tầng phục vụ (vLLM/SGLang). Ở đây chỉ làm phần **quyết định lúc huấn
> luyện ảnh hưởng gì tới lúc phục vụ** — rồi bàn giao.

In [3]:
import json, os, pathlib, sys
sys.path.insert(0, str(pathlib.Path.cwd() / "src"))
sys.path.insert(0, str(pathlib.Path.cwd().parent / "src"))

from labkit import evaluate as ev, generate, report
from labkit.config import get_tier

ROOT = pathlib.Path.cwd() if (pathlib.Path.cwd() / "data").exists() else pathlib.Path.cwd().parent
TIER = get_tier(os.environ.get("COMPUTE_TIER", "T4"))

def load_jsonl(p):
    return [json.loads(l) for l in open(p, encoding="utf-8") if l.strip()]

# EVAL_LIMIT=0 (unset) means the FULL set here, exactly as it does in NB2 and NB5.
# It used to default to 20, so an unabridged run silently scored this notebook's
# no-regression assert on 20 of 50 items while every other notebook used all of them.
EVAL_LIMIT = int(os.environ.get("EVAL_LIMIT", "0") or 0)
target = load_jsonl(ROOT / "data" / "eval_target.jsonl")
if EVAL_LIMIT:
    target = target[:EVAL_LIMIT]
print(f"scoring merge check on {len(target)} target items")

scoring merge check on 50 target items


## 1. Điểm TRƯỚC merge

In [4]:
from peft import PeftModel

model, tok = generate.load_base(TIER)
model = PeftModel.from_pretrained(model, str(ROOT / "adapters" / "correct"))
preds, _ = generate.generate_batch(model, tok, [r["input"] for r in target],
                                   system=generate.NAIVE_PROMPT)
before = sum(ev.triage_field_accuracy(p, r["label"]) for p, r in zip(preds, target)) / len(target)
print(f"trước merge: {before:.4f}")

config.json:   0%|          | 0.00/2.76k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/15.7k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 20.0MB            

tokenizer.json: downloading bytes:           |  0.00B            

added_tokens.json:   0%|          | 0.00/904 [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/426 [00:00<?, ?it/s]

[transformers] `causal_conv1d_fn` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `chunk_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.
[transformers] `causal_conv1d_update` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `fused_recurrent_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.


  [generate] batch 1/13      9s elapsed  ~  104s left
  [generate] batch 2/13     14s elapsed  ~   76s left
  [generate] batch 3/13     19s elapsed  ~   62s left
  [generate] batch 4/13     24s elapsed  ~   55s left
  [generate] batch 5/13     29s elapsed  ~   47s left
  [generate] batch 6/13     35s elapsed  ~   41s left
  [generate] batch 7/13     40s elapsed  ~   34s left
  [generate] batch 8/13     44s elapsed  ~   28s left
  [generate] batch 9/13     50s elapsed  ~   22s left
  [generate] batch 10/13     55s elapsed  ~   16s left
  [generate] batch 11/13     60s elapsed  ~   11s left
  [generate] batch 12/13     65s elapsed  ~    5s left
  [generate] batch 13/13     71s elapsed  ~    0s left
  [generate] done: 50 prompts in 71s
trước merge: 0.9700


## 2. Merge và đo lại

**Assert bắt buộc.** Merge là phép toán chính xác về mặt lý thuyết, nhưng dtype khi
gộp có thể làm tụt điểm. Nếu tụt, đừng deploy — hãy tìm hiểu vì sao.

In [5]:
merged = model.merge_and_unload()
preds_m, _ = generate.generate_batch(merged, tok, [r["input"] for r in target],
                                     system=generate.NAIVE_PROMPT)
after = sum(ev.triage_field_accuracy(p, r["label"]) for p, r in zip(preds_m, target)) / len(target)
delta = after - before
print(f"sau merge:   {after:.4f}   (Δ {delta:+.4f})")

TOL = 0.01
assert delta >= -TOL, (
    f"điểm TỤT {abs(delta):.4f} sau merge (ngưỡng {TOL}). Kiểm tra dtype lúc merge; "
    "với DoRA cần PEFT ≥ 0.10 để gộp đúng vector magnitude (deck §23)."
)

SAVE_MERGED_CHECKPOINT = False  # Bonus chỉ cần phép đo merge; tránh ghi toàn bộ base.
if SAVE_MERGED_CHECKPOINT:
    out = ROOT / "adapters" / "merged"
    merged.save_pretrained(out); tok.save_pretrained(out)
report.write_json({"before_merge": before, "after_merge": after, "delta": delta,
                   "tolerance": TOL, "n": len(target)},
                  "merge_check.json", results_dir=ROOT / "results")
del merged, model; generate.free_memory()

  [generate] batch 1/13      5s elapsed  ~   64s left
  [generate] batch 2/13     12s elapsed  ~   66s left
  [generate] batch 3/13     15s elapsed  ~   51s left
  [generate] batch 4/13     19s elapsed  ~   42s left
  [generate] batch 5/13     22s elapsed  ~   35s left
  [generate] batch 6/13     26s elapsed  ~   30s left
  [generate] batch 7/13     29s elapsed  ~   25s left
  [generate] batch 8/13     32s elapsed  ~   20s left
  [generate] batch 9/13     35s elapsed  ~   15s left
  [generate] batch 10/13     39s elapsed  ~   12s left
  [generate] batch 11/13     42s elapsed  ~    8s left
  [generate] batch 12/13     45s elapsed  ~    4s left
  [generate] batch 13/13     48s elapsed  ~    0s left
  [generate] done: 50 prompts in 48s
sau merge:   0.9700   (Δ +0.0000)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

## 3. Một base, nhiều adapter — hoán đổi theo request

Đây là lập luận kinh tế của LoRA ở deck §23: base nằm trong VRAM một lần, mỗi khách
hàng/tác vụ là một adapter vài chục MB.

In [6]:
model, tok = generate.load_base(TIER)
model = PeftModel.from_pretrained(model, str(ROOT / "adapters" / "correct"),
                                  adapter_name="correct")
available = ["correct"]
for extra in ("attn_only",):
    d = ROOT / "adapters" / extra
    if d.exists():
        model.load_adapter(str(d), adapter_name=extra)
        available.append(extra)

print("adapter đang nạp:", available)
ticket = target[0]["input"]
hotswap_outputs = {}
base_identity = id(model.get_base_model())
for name in available:
    model.set_adapter(name)
    out, _ = generate.generate_batch(model, tok, [ticket], system=generate.NAIVE_PROMPT)
    assert id(model.get_base_model()) == base_identity
    hotswap_outputs[name] = out[0]
    print(f"\n[{name}] -> {out[0][:140]}")
assert len(available)>=2
report.write_json({"adapters":available,"same_base":True,"ticket":ticket,"outputs":hotswap_outputs},"hotswap_check.json",results_dir=ROOT / "results")
del model
generate.free_memory()


Loading weights:   0%|          | 0/426 [00:00<?, ?it/s]

adapter đang nạp: ['correct', 'attn_only']
  [generate] batch 1/1      5s elapsed  ~    0s left
  [generate] done: 1 prompts in 5s

[correct] -> {"intent": "doi_tra", "urgency": "cao", "product": "chuột không dây", "sentiment": "tich_cuc"}
  [generate] batch 1/1      3s elapsed  ~    0s left
  [generate] done: 1 prompts in 3s

[attn_only] -> {"intent": "doi_tra", "urgency": "cao", "product": "chuột không dây", "sentiment": "tich_cuc"}


## ✅ Checkpoint NB6
- [ ] `results/merge_check.json` — điểm sau merge không tụt quá 0.01
- [ ] Đã hoán đổi ≥2 adapter trên **cùng một** base đang nạp

→ Ngày 20 (Model Serving) là nơi biến việc này thành một endpoint đa người thuê.

In [7]:
# @title Xuất bằng chứng bonus (chạy lại sau upload để lấy cả link)
from zipfile import ZipFile,ZIP_DEFLATED
bonus_zip=ROOT.parent / "lab21_bonus_evidence.zip"
with ZipFile(bonus_zip,"w",ZIP_DEFLATED) as z:
    for name in ["merge_check.json","hotswap_check.json","hf_hub.json"]:
        path=ROOT / "results" / name
        if path.exists():z.write(path,"results/"+name)
print("ZIP bonus:",bonus_zip)
from google.colab import files
files.download(str(bonus_zip))


ZIP bonus: /content/lab21_bonus_evidence.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Upload adapter công khai
Tạo token Write tại https://huggingface.co/settings/tokens. Run All sẽ yêu cầu token trong ô nhập ẩn; code chỉ upload hai file adapter và model card, không upload token hoặc trọng số base.


In [8]:
# @title B5 — upload correct lên Hugging Face công khai
from getpass import getpass
from huggingface_hub import HfApi
import shutil
hf_token=getpass("HF token có quyền write (không gửi vào chat): ")
try:
    api=HfApi(token=hf_token)
    username=api.whoami()["name"]
    repo_id=username+"/lab21-qwen35-4b-triage-lora"
    upload_dir=ROOT / "hub_export"
    upload_dir.mkdir(exist_ok=True)
    for name in ["adapter_config.json","adapter_model.safetensors"]:
        shutil.copy2(ROOT / "adapters/correct" / name,upload_dir / name)
    verdict=json.loads((ROOT / "results/verdict.json").read_text(encoding="utf-8"))
    card="---\nbase_model: unsloth/Qwen3.5-4B\nlibrary_name: peft\nlanguage:\n- vi\ntags:\n- lora\n- text-generation\n---\n# Lab21 Vietnamese ticket triage adapter\n\nEducational LoRA adapter; requires the original base model. Trained on the lab's 225-ticket split, r=16, alpha=32, LR=1e-4, 30 optimizer steps.\n\nEvaluation: "+json.dumps(verdict["comparison"],ensure_ascii=False,indent=2)+"\n\nRegression gate: "+json.dumps(verdict["verdict"],ensure_ascii=False)+"\n\nThis adapter failed the general-capability gate. It is a lab artifact, not a production validation. Refer to the base model terms when using it.\n"
    (upload_dir / "README.md").write_text(card,encoding="utf-8")
    repo_url=api.create_repo(repo_id=repo_id,repo_type="model",private=False,exist_ok=True)
    existing=api.model_info(repo_id)
    assert not existing.private, "Repo hiện có là private: đổi sang public trong HF rồi chạy lại."
    commit=api.upload_folder(repo_id=repo_id,repo_type="model",folder_path=str(upload_dir),allow_patterns=["adapter_config.json","adapter_model.safetensors","README.md"],commit_message="Publish Lab21 correct LoRA adapter and measured limitations")
    verified=api.model_info(repo_id,revision=commit.oid)
    names={f.rfilename for f in verified.siblings}
    assert not verified.private and {"adapter_config.json","adapter_model.safetensors","README.md"} <= names
    report.write_json({"repo_id":repo_id,"url":str(repo_url),"commit":commit.oid,"public":True,"files_verified":True},"hf_hub.json",results_dir=ROOT / "results")
    print("ĐÃ UPLOAD VÀ XÁC MINH:",repo_url)
finally:
    hf_token=None
    if "api" in globals():del api


HF token có quyền write (không gửi vào chat): ··········
ĐÃ UPLOAD VÀ XÁC MINH: https://huggingface.co/huyhaithanh51/lab21-qwen35-4b-triage-lora


In [9]:
# @title Xuất bằng chứng bonus (chạy lại sau upload để lấy cả link)
from zipfile import ZipFile,ZIP_DEFLATED
bonus_zip=ROOT.parent / "lab21_bonus_evidence.zip"
with ZipFile(bonus_zip,"w",ZIP_DEFLATED) as z:
    for name in ["merge_check.json","hotswap_check.json","hf_hub.json"]:
        path=ROOT / "results" / name
        if path.exists():z.write(path,"results/"+name)
print("ZIP bonus:",bonus_zip)
from google.colab import files
files.download(str(bonus_zip))


ZIP bonus: /content/lab21_bonus_evidence.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>